# 🍳 Kitchen Manager AI — Web UI (Gradio)

A simple browser-based UI for the meal-plan generator. Backed by an **open-source LLM** (default: `meta-llama/Llama-3.3-70B-Instruct`) served via HuggingFace's Inference API.

**Run order:**
1. Cell 1 — Install dependencies
2. Cell 2 — Imports, HuggingFace token, model setup
3. Cell 3 — Sample inventory + system prompt + report-generation function
4. Cell 4 — Launch the Gradio web UI

When Cell 4 runs, Gradio prints a `https://...gradio.live` link. Click it to use the UI in your browser.

**The UI lets you:**
- Upload `ingredients.csv`, `utensils.csv`, `equipment.csv` *(or)* use bundled sample data
- Answer Q1 (number of people) and Q2 (dietary preferences, multi-select + custom)
- Click **Generate Meal Plan** → renders the full 7-section markdown report and offers a download

## Cell 1 — Install dependencies

In [ ]:
!pip install gradio huggingface_hub pandas --quiet

## Cell 2 — Imports, HuggingFace token, model setup
Get a free token from https://huggingface.co/settings/tokens. Some gated models (e.g., Llama 3.3) require accepting the model license on its HF page first.

In [ ]:
import os
import io
import tempfile
import pandas as pd
import gradio as gr
from getpass import getpass
from huggingface_hub import InferenceClient

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("🔑 Enter your HuggingFace token: ")

# Default open-source model. Swap as you like:
#   - "meta-llama/Llama-3.3-70B-Instruct"          (default — strong, gated)
#   - "Qwen/Qwen2.5-72B-Instruct"                  (very strong, instruction-following)
#   - "mistralai/Mixtral-8x7B-Instruct-v0.1"       (open weights, no gating)
#   - "meta-llama/Meta-Llama-3.1-8B-Instruct"      (smaller, faster)
#   - "Qwen/Qwen2.5-32B-Instruct"
MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"

client = InferenceClient(model=MODEL_ID, token=os.environ["HF_TOKEN"])
print(f"✅ HuggingFace InferenceClient initialized with model: {MODEL_ID}")

✅ HuggingFace InferenceClient initialized with model: meta-llama/Llama-3.3-70B-Instruct


## Cell 3 — Sample data, system prompt, generation function

In [ ]:
SAMPLE_INGREDIENTS = """ingredient_name,quantity,unit
Basmati Rice,2000,grams
Whole Wheat Flour,1500,grams
Toor Dal,800,grams
Chickpeas,500,grams
Paneer,400,grams
Tomatoes,1500,grams
Onions,1200,grams
Potatoes,2000,grams
Spinach,500,grams
Carrots,600,grams
Bell Pepper,400,grams
Garlic,150,grams
Ginger,100,grams
Green Chili,50,grams
Cucumber,400,grams
Yogurt,500,grams
Milk,2000,milliliters
Eggs,12,pieces
Butter,200,grams
Olive Oil,500,milliliters
Mustard Oil,300,milliliters
Cumin Seeds,80,grams
Turmeric,60,grams
Coriander Powder,80,grams
Garam Masala,50,grams
Salt,500,grams
Black Pepper,40,grams
Honey,250,grams
Lemon,6,pieces
Oats,500,grams
Bread,12,slices
Banana,8,pieces
Apple,6,pieces
Almonds,200,grams
"""

SAMPLE_UTENSILS = """utensil_name,available
Frying Pan,yes
Saucepan,yes
Pressure Cooker,yes
Wok / Kadhai,yes
Mixing Bowls,yes
Knife Set,yes
Cutting Board,yes
Baking Tray,yes
Rolling Pin,yes
Tawa / Flat Pan,yes
Blender,yes
Whisk,yes
Strainer,yes
Grater,yes
Ladle,yes
Spatula,yes
"""

SAMPLE_EQUIPMENT = """equipment_type,available
Gas Flame,yes
Electric Burner,no
Oven,yes
Microwave,yes
Air Fryer,no
Toaster,yes
Refrigerator,yes
"""

DIET_OPTIONS = [
    "None / No restrictions",
    "Keto",
    "Intermittent Fasting",
    "Navratri (no onion, garlic, non-veg, certain grains)",
    "Vegan",
    "Vegetarian",
]

SYSTEM_PROMPT = """You are an expert Kitchen Manager AI. Your job is to analyze a user's kitchen inventory and generate a complete, structured weekly meal plan report in a single output.

The runtime inputs (number of people, dietary preferences) and the inventory CSVs have already been provided in the user message. Do NOT ask any follow-up questions. Generate the FULL report in one shot.

## STEP 2 — Parse the CSV Inventory
The user has provided three CSV blocks:
1. Ingredients — columns: ingredient_name, quantity, unit
2. Kitchen Utensils — columns: utensil_name, available (yes/no)
3. Cooking Equipment / Burners — columns: equipment_type (e.g., gas flame, electric burner, oven, microwave, air fryer), available (yes/no)

If any column is missing or ambiguous, make a reasonable assumption and explicitly state it under the \"Assumptions Made\" section.

## STEP 3 — Generate the Full Report
Produce the following sections in order with clear markdown headers, tables, and bullet points throughout.

### 📋 Section 0: Assumptions Made
List any assumptions about missing data, ambiguous quantities, or unclear equipment entries.

### 🍳 Section 1: Available Dish Possibilities
Based strictly on the provided ingredients, utensils, and equipment:
- List all viable dishes that can be made
- Group by: Breakfast / Lunch / Dinner / Snacks
- Cuisine style: Global / International — suggest dishes from diverse world cuisines where ingredients allow
- Flag any dish requiring an ingredient that is low in quantity (< 2 servings remaining)
- Apply all dietary preferences as HARD filters — never suggest dishes that violate them

### 📅 Section 2: 7-Day Meal Plan (Monday – Sunday)
For each day provide: Breakfast, Lunch, Dinner (and optionally a Snack).
Format as a table:
| Day | Meal | Dish | Prep Time | Cooking Method | Ingredients & Quantities Used |
|-----|------|------|-----------|----------------|-------------------------------|
In the **\"Ingredients & Quantities Used\"** column, list every ingredient consumed in that meal followed by the exact quantity in parentheses, scaled for the number of people specified. Use the SAME UNITS as the inventory CSV (grams, milliliters, pieces, slices). Example for a family of 4: `Basmati Rice (300g), Tomatoes (200g), Onion (100g), Cumin Seeds (3g), Olive Oil (15ml)`.
Rules:
- Do NOT repeat the same dish more than twice in the week
- Respect all dietary preferences strictly
- Only suggest dishes cookable with the listed utensils and equipment
- Scale all portions for the number of people specified
- Quantities listed in this section MUST be the basis for the consumption math used in Section 5 (Shopping List)

### 🥗 Section 3: Nutritional Analysis
For each day provide an approximate daily nutritional summary:
| Day | Calories | Protein (g) | Carbs (g) | Fats (g) | Fiber (g) | Key Micronutrients |
|-----|----------|-------------|-----------|----------|-----------|--------------------|
At the bottom include weekly averages across all nutrients and the disclaimer: \"Values are approximate and based on standard serving sizes. Consult a nutritionist for medical-grade guidance.\"

### ⚠️ Section 4: Nutrient Deficits
Compare against standard DRI benchmarks for the specified number of people.
| Nutrient | Avg Daily Amount | Recommended Daily Amount | Gap | Suggested Fix |
|----------|-----------------|--------------------------|-----|---------------|
Flag if a dietary preference (e.g., Vegan, Keto) is the structural cause of a deficit.

### 🛒 Section 5: Shopping List
Calculate ingredient consumption across the full 7-day plan and compare against current inventory.

**5A — Ingredients to Buy (Deficit / Running Low)**
| # | Ingredient | Quantity Needed | Unit | Priority (High / Medium / Low) | Category |
|---|------------|-----------------|------|-------------------------------|----------|
Categories: Vegetables, Fruits, Dairy, Grains & Legumes, Proteins, Spices & Condiments, Oils & Fats, Beverages, Other.
Priority: High = needed within first 2 days; Medium = days 3–5; Low = days 6–7 or optional.

**5B — Ingredients in Excess**
| Ingredient | Current Qty | Qty Used in Plan | Remaining | Suggestion |
|------------|-------------|------------------|-----------|------------|
Suggestions like: \"Use in Week 2 planning\", \"Store properly — shelf life ~X days\", \"Consider gifting or composting\".

### ✅ Section 6: Final Summary
5–8 sentences covering: dietary approach taken, cuisine diversity achieved, overall nutritional balance rating (Good / Moderate / Needs Improvement) with one-line justification, and top 2–3 actionable recommendations.

## FORMATTING RULES
- Use markdown throughout
- Every section has a header with an emoji
- All tables must be properly aligned
- Never skip a section even if data is limited — state \"Insufficient data to populate this section\" and explain why
- Do not ask any follow-up questions — generate the full report in one shot
"""


def _read_csv(path_or_text, is_text=False):
    if is_text:
        return pd.read_csv(io.StringIO(path_or_text))
    return pd.read_csv(path_or_text)


def generate_report(ingredients_file, utensils_file, equipment_file, use_sample, people, diets, other_diet, progress=gr.Progress()):
    """Main pipeline called by the Gradio button.
    Returns: (status_md, report_md, download_path_or_None).
    """
    progress(0.1, desc="Reading inventory...")

    # 1. Load inventory (sample or uploaded)
    try:
        if use_sample:
            ingredients_df = _read_csv(SAMPLE_INGREDIENTS, is_text=True)
            utensils_df = _read_csv(SAMPLE_UTENSILS, is_text=True)
            equipment_df = _read_csv(SAMPLE_EQUIPMENT, is_text=True)
        else:
            missing = [n for n, f in [("ingredients", ingredients_file), ("utensils", utensils_file), ("equipment", equipment_file)] if f is None]
            if missing:
                return (f"❌ Missing CSV upload(s): **{', '.join(missing)}**. Either upload all three or tick *Use bundled sample inventory*.", "", None)
            ingredients_df = _read_csv(ingredients_file)
            utensils_df = _read_csv(utensils_file)
            equipment_df = _read_csv(equipment_file)
    except Exception as e:
        return (f"❌ Failed to parse CSVs: `{e}`", "", None)

    # 2. Validate runtime inputs
    if not str(people).strip():
        return ("❌ Please enter the number of people (Q1).", "", None)

    selected_diets = list(diets) if diets else []
    if other_diet and other_diet.strip():
        selected_diets.append(f"Other: {other_diet.strip()}")
    if not selected_diets:
        selected_diets = ["None / No restrictions"]

    progress(0.25, desc=f"Calling {MODEL_ID}...")

    # 3. Build user message
    user_msg = f"""Runtime inputs and inventory data are below. Generate the full Kitchen Manager AI report now.

**Number of people:** {people}
**Dietary preferences:** {', '.join(selected_diets)}

**Ingredients (CSV):**
```csv
{ingredients_df.to_csv(index=False)}```

**Utensils (CSV):**
```csv
{utensils_df.to_csv(index=False)}```

**Cooking Equipment (CSV):**
```csv
{equipment_df.to_csv(index=False)}```

Produce all sections (0–6) in one response.
"""

    # 4. Call HF Inference
    try:
        response = client.chat_completion(
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            max_tokens=8000,
            temperature=0.7,
            top_p=0.9,
        )
        report_md = response.choices[0].message.content
    except Exception as e:
        return (f"❌ LLM call failed: `{e}`", "", None)

    progress(0.9, desc="Saving report...")

    # 5. Save markdown for download
    out_path = os.path.join(tempfile.gettempdir(), "kitchen_meal_plan_report.md")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(report_md)

    usage = getattr(response, "usage", None)
    status = f"✅ Report ready ({len(report_md):,} chars"
    if usage is not None:
        status += f", tokens: in={usage.prompt_tokens}, out={usage.completion_tokens}"
    status += f", model: `{MODEL_ID}`)."

    progress(1.0, desc="Done")
    return (status, report_md, out_path)


print("✅ generate_report() defined.")

✅ generate_report() defined.


## Cell 4 — Launch the Gradio web UI
When this cell runs, Gradio prints a public `https://...gradio.live` URL. Click it to open the UI in your browser.

In [ ]:
with gr.Blocks(title="Kitchen Manager AI", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🍳 Kitchen Manager AI
        Upload your kitchen inventory + answer 2 quick questions → get a full 7-day meal plan,
        nutrition analysis, and a categorized shopping list.
        """
    )

    with gr.Row():
        # LEFT — Inventory uploads
        with gr.Column(scale=1):
            gr.Markdown("### 📂 Step 1 — Inventory CSVs")
            ingredients_file = gr.File(label="ingredients.csv", file_types=[".csv"], type="filepath")
            utensils_file = gr.File(label="utensils.csv", file_types=[".csv"], type="filepath")
            equipment_file = gr.File(label="equipment.csv", file_types=[".csv"], type="filepath")
            use_sample = gr.Checkbox(
                label="Use bundled sample inventory instead",
                value=False,
                info="Tick this to skip uploads and try the demo with the included sample data.",
            )

        # RIGHT — Runtime questions
        with gr.Column(scale=1):
            gr.Markdown("### 📋 Step 2 — Runtime Questions")
            people = gr.Textbox(
                label="Q1. How many people is this meal plan for?",
                value="2",
                placeholder="e.g., 1, 2, family of 4",
            )
            diets = gr.CheckboxGroup(
                label="Q2. Dietary preferences (select all that apply)",
                choices=DIET_OPTIONS,
                value=["None / No restrictions"],
            )
            other_diet = gr.Textbox(
                label="Other (specify if not listed)",
                placeholder="e.g., Gluten-free, Halal, Diabetic, Low-FODMAP...",
            )

    generate_btn = gr.Button("🍳 Generate Meal Plan", variant="primary", size="lg")

    gr.Markdown("---")
    status = gr.Markdown(value="*Waiting for input. Fill out the form above and click Generate.*")
    download = gr.File(label="⬇️ Download report (.md)", visible=False, interactive=False)

    gr.Markdown("## 📄 Generated Report")
    output_md = gr.Markdown(value="*Your generated report will appear here.*")

    def _run(ingr, utens, equip, use_s, ppl, dts, other):
        status_msg, report, dl_path = generate_report(ingr, utens, equip, use_s, ppl, dts, other)
        return (
            status_msg,
            report if report else "*(no report)*",
            gr.update(value=dl_path, visible=dl_path is not None),
        )

    generate_btn.click(
        _run,
        inputs=[ingredients_file, utensils_file, equipment_file, use_sample, people, diets, other_diet],
        outputs=[status, output_md, download],
    )

    gr.Markdown(
        "<sub>Powered by an open-source LLM via HuggingFace Inference API. "
        "Nutritional values are approximate — consult a nutritionist for medical-grade guidance.</sub>"
    )

# In Colab use share=True so you get a public *.gradio.live URL.
demo.launch(share=True, debug=False)

/tmp/ipykernel_22735/3544411987.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Kitchen Manager AI", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ce746dcd08c009cb23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!git clone https://github.com/RISHI1328/Kitchen-AI-Planner.git

Cloning into 'Kitchen-AI-Planner'...


In [ ]:
%cd Kitchen-AI-Planner

/content/Kitchen-AI-Planner


In [ ]:
!ls /content

Kitchen-AI-Planner  sample_data
